# Data pre-processing

### Import dataset

In [1]:
import pandas as pd
import  numpy as np
# From 2019
df_2019 = pd.read_excel("./data/2019_updated.xls")

# From 2017
df_labeled = pd.read_excel("./data/MichelleCoding3KAR_02_19_2025.xlsx")
df_labeled = df_labeled[df_labeled["AR"].notnull()]

In [2]:
df_labeled['year'] = df_labeled['Unnamed: 3'].astype(str).str[:4]
df_labeled.year.value_counts()

2017    1001
2020       1
Name: year, dtype: int64

### Select top senators with at least 20 tweets

In [4]:
def get_labels(df, col_name):
    conditions = [
        df[col_name] == 1,
        df[col_name] == 2,
        df[col_name] == 3
    ]
    choices = ['Problem', 'Solution', 'Other']
    return np.select(conditions, choices, default=np.NAN)

top_20_senators = df_labeled[df_labeled.groupby("author_id")["id"].transform("count") > 20]
top_20_senators["Annelise_label"] = get_labels(top_20_senators, "AR")
top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].astype("category")

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/2464385180.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["Annelise_label"] = get_labels(top_20_senators, "AR")
/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/2464385180.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].astype("category")


In [7]:
baseline_senator = "Rick Scott"

# Ensure 'author_id' is categorical with Rick Scott as the reference category
top_20_senators["author_id"] = top_20_senators["author_id"].astype("category")
top_20_senators["author_id"] = top_20_senators["author_id"].cat.set_categories(
    sorted(top_20_senators["author_id"].unique(), key=lambda x: x != baseline_senator)
)
top_20_senators.author_id.value_counts()

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/1365189477.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["author_id"] = top_20_senators["author_id"].astype("category")
/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/1365189477.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["author_id"] = top_20_senators["author_id"].cat.set_categories(


Rick Scott            147
Rob Portman            34
Elizabeth Warren       30
Lindsey Graham         28
Jeff Merkley           28
Patrick Leahy          26
Chuck Schumer          26
John Cornyn            23
Sheldon Whitehouse     22
Chris Murphy           21
Ted Cruz               21
Name: author_id, dtype: int64

In [6]:
top_20_senators[["author_id"]].head(1)

,author_id
0,Lindsey Graham


# Correlation test

### Logistic regression

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


import pandas as pd
import statsmodels.api as sm

# pre-processing stream labels
top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].astype("category")
top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].cat.reorder_categories(['Problem', 'Solution', 'Other'], ordered=True)
# category_mapping = dict(enumerate(top_20_senators["Annelise_label"].cat.categories))
colunn_mapping = {0: "Solution", 1:"Other"}

# pre-processing author_id
baseline_senator = "Rick Scott"
top_20_senators["author_id"] = top_20_senators["author_id"].astype("category")
top_20_senators["author_id"] = top_20_senators["author_id"].cat.set_categories(
    sorted(top_20_senators["author_id"].unique(), key=lambda x: x != baseline_senator)
)
df_encoded = pd.get_dummies(top_20_senators[["author_id"]], drop_first=True)

# Add an intercept column manually (statsmodels does not add it by default)
df_encoded = sm.add_constant(df_encoded)

# Fit multinomial logistic regression (MNLogit)
model = sm.MNLogit(top_20_senators['Annelise_label'], df_encoded)
result = model.fit(method='newton', maxiter=1000)

# Extract coefficients
coef_df = pd.DataFrame(result.params).rename(columns=colunn_mapping)
odds_df = np.exp(coef_df)

Optimization terminated successfully.
         Current function value: 0.903973
         Iterations 7


/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/249278658.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].astype("category")
/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_89280/249278658.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_20_senators["Annelise_label"] = top_20_senators["Annelise_label"].cat.reorder_categories(['Problem', 'Solution', 'Other'], ordered=True)
/var/folders/_5/c3vpzww13z3

In [13]:
print(result.summary())

                          MNLogit Regression Results                          
Dep. Variable:         Annelise_label   No. Observations:                  406
Model:                        MNLogit   Df Residuals:                      384
Method:                           MLE   Df Model:                           20
Date:                Mon, 17 Mar 2025   Pseudo R-squ.:                  0.1769
Time:                        16:41:24   Log-Likelihood:                -367.01
converged:                       True   LL-Null:                       -445.88
Covariance Type:            nonrobust   LLR p-value:                 2.058e-23
     Annelise_label=Solution       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            1.3564      0.272      4.987      0.000       0.823       1.890
author_id_Lindsey Graham        -0.4914      0.502     -0.980      0.327     

In [14]:
odds_df

,Solution,Other
const,3.882353,3.764706
author_id_Lindsey Graham,0.611742,0.033203
author_id_John Cornyn,0.343434,0.398438
author_id_Jeff Merkley,0.021465,0.022135
author_id_Rob Portman,0.257576,0.113839
author_id_Chris Murphy,0.160985,0.265625
author_id_Ted Cruz,0.257576,0.863281
author_id_Sheldon Whitehouse,0.070248,0.193182
author_id_Elizabeth Warren,0.122010,0.027961
author_id_Patrick Leahy,0.064394,0.531250


### Chi-squared

In [47]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(top_20_senators['author_id'], top_20_senators['Annelise_label'])

# Step 2: Perform the chi-squared test
chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

# Step 3: Interpret the result
print(f"Chi-squared Statistic (rounded to 2 decimals): {round(chi2_stat, 2)}")
print(f"P-value (rounded to 6 decimals): {round(p_value,6)}")

if p_value < 0.05:
    print("Reject the null hypothesis: The variables are dependent (correlated).")
else:
    print("Fail to reject the null hypothesis: The variables are independent (not correlated).")

Chi-squared Statistic (rounded to 2 decimals): 146.48
P-value (rounded to 6 decimals): 0.0
Reject the null hypothesis: The variables are dependent (correlated).


In [48]:
contingency_table

Annelise_label,Problem,Solution,Other
author_id,,,
Chris Murphy,8,5,8
Chuck Schumer,19,6,1
Elizabeth Warren,19,9,2
Jeff Merkley,24,2,2
John Cornyn,6,8,9
Lindsey Graham,8,19,1
Patrick Leahy,8,2,16
Rick Scott,17,66,64
Rob Portman,14,14,6
